# Detection Results Viewer

The test period is 2023–2024, with 459 merged target events and a seven-day warning horizon. Non-operation includes maintenance, communication, curtailment, non-Full-Performance status, target-event intervals, and the selected environmental buffer.


In [1]:
from pathlib import Path
import asyncio
import importlib
import sys
import traceback

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import main_detect_residual_baseline as detection_core
importlib.reload(detection_core)
import scripts.analysis.detection_results_viewer_data as viewer_data
importlib.reload(viewer_data)
from scripts.analysis.detection_results_viewer_data import (
    DEFAULT_SETTINGS, METHOD_LABELS, available_settings, list_runs,
    load_coverage_comparison, load_non_operational_intervals, load_result, load_timeline,
    plot_coverage_comparison, plot_timeline, visible_alarm_intervals,
)

runs = list_runs(PROJECT_ROOT)
if not runs:
    raise FileNotFoundError('No GRU experiment found under results/kelmarsh/experiments')

method_dropdown = widgets.Dropdown(options=[(label, key) for key, label in METHOD_LABELS.items()], description='Method')
run_dropdown = widgets.Dropdown(options=runs, value=runs[-1], description='Run')
setting_dropdown = widgets.Dropdown(description='Setting', layout=widgets.Layout(width='650px'))
buffer_dropdown = widgets.Dropdown(options=[('None', 0), ('+/- 1 h', 1), ('+/- 3 h', 3), ('+/- 6 h', 6)], value=6, description='Env buffer')
non_operational_rule_dropdown = widgets.Dropdown(options=[('Keep TA; split other alarms', 'remaining_consecutive'), ('Any overlap', 'overlap'), ('Fully contained', 'contained')], value='remaining_consecutive', description='Episode rule')

def update_settings(*_):
    settings = available_settings(PROJECT_ROOT, run_dropdown.value, method_dropdown.value)
    setting_dropdown.options = settings
    preferred = DEFAULT_SETTINGS[method_dropdown.value]
    setting_dropdown.value = preferred if preferred in settings else (settings[0] if settings else None)

def update_method_rule(*_):
    non_operational_rule_dropdown.value = 'remaining_consecutive'

method_dropdown.observe(update_settings, names='value')
method_dropdown.observe(update_method_rule, names='value')
run_dropdown.observe(update_settings, names='value')
update_settings()
display(widgets.VBox([widgets.HBox([method_dropdown, run_dropdown, setting_dropdown]), widgets.HBox([buffer_dropdown, non_operational_rule_dropdown])]))


## Non-Operational Coverage


In [2]:
coverage_button = widgets.Button(description='Compare coverage')
coverage_output = widgets.Output()

def show_coverage(_):
    with coverage_output:
        clear_output(wait=True)
        comparison = load_coverage_comparison(PROJECT_ROOT)
        fig = plot_coverage_comparison(comparison)
        display(fig)
        plt.close(fig)
        display(comparison.assign(coverage=comparison['coverage'].map(lambda value: f'{value:.2%}')))

coverage_button.on_click(show_coverage)
display(widgets.VBox([coverage_button, coverage_output]))


## Key Metrics

Figures are for all six turbines and the selected environmental buffer; turbine and year affect only the timeline.


In [3]:
metrics_button = widgets.Button(description='Load metrics', button_style='primary')
metrics_output = widgets.Output()
metrics_status = widgets.HTML()

async def render_metrics(selection):
    run, method, setting, buffer_hours, rule = selection
    try:
        metadata, values, thresholds = await asyncio.to_thread(
            load_result, PROJECT_ROOT, run, method, setting, buffer_hours, rule
        )
        with metrics_output:
            clear_output(wait=True)
            display(Markdown(f'### {METHOD_LABELS[method]} · {setting} · buffer +/-{buffer_hours} h · {rule}'))
            display(pd.DataFrame([{'turbines': ', '.join(metadata.get('turbines', [])),
                                   'best_epoch': metadata.get('best_epoch'), 'setting': setting}]))
            rows = [
                ('Target events detected', f"{values['detected_events']}/{values['total_events']}"),
                ('Event recall', f"{values['event_recall']:.1%}"),
                ('Missed events', values['missed_events']),
                ('Alarm episodes', values['total_alarm_episodes']),
                ('True-alarm episodes', values['true_alarm_episodes']),
                ('Operational false-alarm episodes', values['operational_false_alarm_episodes']),
                ('Ignored non-operational episodes' if rule != 'remaining_consecutive' else 'Suppressed source episodes', values['ignored_non_operational_episodes']),
                ('Operational precision', f"{values['operational_precision']:.1%}"),
                ('Operational false-alarm rate', f"{values['operational_false_alarm_rate']:.1%}"),
                ('F1', f"{values['f1']:.1%}"),
            ]
            if 'min_operational_consecutive' in values:
                rows.insert(7, ('Minimum c (from Setting)', values['min_operational_consecutive']))
            display(pd.DataFrame(rows, columns=['Metric', 'Value']).set_index('Metric'))
            if not thresholds.empty:
                columns = ['target', 'k', 'reference_value', 'decision_threshold', 'window_size',
                           'threshold_quantile', 'threshold', 'center', 'scale',
                           'healthy_weibull_shape', 'healthy_weibull_scale',
                           'weibull_shape', 'weibull_scale']
                display(Markdown('### Detection Thresholds / Reference Values'))
                display(thresholds[[col for col in columns if col in thresholds.columns]])
        metrics_status.value = 'Ready'
    except Exception:
        metrics_status.value = 'Failed; see error below'
        metrics_output.append_stdout(traceback.format_exc())
    finally:
        metrics_button.disabled = False
        metrics_button.description = 'Load metrics'

def show_metrics(_):
    if metrics_button.disabled:
        return
    if setting_dropdown.value is None:
        metrics_status.value = 'No complete result for this setting'
        return
    selection = (run_dropdown.value, method_dropdown.value, setting_dropdown.value,
                 buffer_dropdown.value, non_operational_rule_dropdown.value)
    metrics_output.clear_output()
    metrics_status.value = 'Loading metrics...'
    metrics_button.disabled = True
    metrics_button.description = 'Loading...'
    asyncio.get_running_loop().create_task(render_metrics(selection))

metrics_button.on_click(show_metrics)
display(widgets.VBox([widgets.HBox([metrics_button, metrics_status]), metrics_output]))


## 2023–2024 Event and Alarm Timeline

The lanes show target events, true-alarm (TA) episodes, and operational false-alarm (FA) episodes. A source episode matching an event horizon remains TA even when it overlaps non-operation. For other episodes, Keep TA; split other alarms removes points touching non-operation, splits the remaining time into separate episodes, and discards runs shorter than c. For BCAD, the c encoded in Setting is used both to generate alarm points and to filter these remaining runs; CUSUM uses 1. Gray shading shows non-operational time. Metrics and timeline are recalculated for the selected rule and buffer; original result files remain unchanged.


In [4]:
turbine_dropdown = widgets.Dropdown(options=[f'Kelmarsh_{i}' for i in range(1, 7)], description='Turbine')
year_dropdown = widgets.Dropdown(options=[2023, 2024], description='Year')
show_non_operational = widgets.Checkbox(value=True, description='Show non-operation')
timeline_button = widgets.Button(description='Draw timeline', button_style='primary')
timeline_output = widgets.Output()
timeline_status = widgets.HTML()

async def render_timeline(selection):
    run, method, setting, turbine, year, buffer_hours, rule, show_shading = selection
    fig = None
    try:
        events, episodes = await asyncio.to_thread(
            load_timeline, PROJECT_ROOT, run, method, setting, turbine, year, buffer_hours, rule
        )
        non_operational = await asyncio.to_thread(
            load_non_operational_intervals, PROJECT_ROOT, turbine, year, buffer_hours
        )
        fig = plot_timeline(
            events, episodes, turbine, year, method, setting, non_operational,
            show_non_operational=show_shading, buffer_hours=buffer_hours,
            non_operational_rule=rule,
        )
        output_dir = PROJECT_ROOT / 'results' / 'kelmarsh' / 'figures' / 'detection_timelines' / 'no_warning'
        output_dir.mkdir(parents=True, exist_ok=True)
        stem = f'{method}_{setting}_{turbine}_{year}_buffer{buffer_hours}h_{rule}_turbine_{int(show_shading)}'
        png_path = output_dir / f'{stem}.png'
        fig.savefig(png_path, dpi=240, bbox_inches='tight')
        fig.savefig(png_path.with_suffix('.pdf'), bbox_inches='tight')
        start, end = pd.Timestamp(year, 1, 1), pd.Timestamp(year + 1, 1, 1)
        ta, fa = visible_alarm_intervals(episodes, start, end)
        with timeline_output:
            clear_output(wait=True)
            display(fig)
            print(f'Target events: {len(events)}; visible TA intervals: {len(ta)}; visible FA intervals: {len(fa)}')
            print(f'Saved: {png_path}')
        timeline_status.value = 'Ready'
    except Exception:
        timeline_status.value = 'Failed; see error below'
        timeline_output.append_stdout(traceback.format_exc())
    finally:
        if fig is not None:
            plt.close(fig)
        timeline_button.disabled = False
        timeline_button.description = 'Draw timeline'

def draw_selected_timeline(_):
    if timeline_button.disabled:
        return
    if setting_dropdown.value is None:
        timeline_status.value = 'No complete result for this setting'
        return
    selection = (run_dropdown.value, method_dropdown.value, setting_dropdown.value,
                 turbine_dropdown.value, year_dropdown.value, buffer_dropdown.value,
                 non_operational_rule_dropdown.value, show_non_operational.value)
    timeline_output.clear_output()
    timeline_status.value = 'Loading timeline...'
    timeline_button.disabled = True
    timeline_button.description = 'Loading...'
    asyncio.get_running_loop().create_task(render_timeline(selection))

timeline_button.on_click(draw_selected_timeline)
display(widgets.VBox([widgets.HBox([turbine_dropdown, year_dropdown, show_non_operational, timeline_button, timeline_status]), timeline_output]))
